# Prism — Fine-tune ResNet-18 — Slot C (pretrained, modern)
**Christine — ResNet-18 Lead**

Trains the **Slot C (pretrained)** model on CIFAR-10 and evaluates it on the test set,
saving `metrics.json`, the confusion matrix, training curves, and the model.

### Colab setup
1. Runtime → Change runtime type → **GPU** (T4).
2. Clone the repo into Colab:
   `!git clone https://github.com/JethroIsHere/prism.git`
   (or upload the project folder and set `PROJECT_ROOT` below).
3. Run the cells top to bottom.


In [ ]:
!git clone https://github.com/JethroIsHere/prism.git /content/prism

fatal: destination path '/content/prism' already exists and is not an empty directory.


In [ ]:
import json, os, sys, time
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

# Point this at the project root (the folder containing duke-custom-cnn/, etc.)
PROJECT_ROOT = "/content/prism" if os.path.isdir("/content/prism") else "."
sys.path.insert(0, os.path.join(PROJECT_ROOT, "christine-resnet18"))
sys.path.insert(0, os.path.join(PROJECT_ROOT, "shared"))

from resnet18_finetune import (build_model, count_parameters, fit, evaluate,
                               plot_curves, confusion_matrix,
                               accuracy_and_macro_f1, plot_confusion_matrix)
from data_utils import load_canonical_splits, CLASS_NAMES, SEED

RESULTS = os.path.join(PROJECT_ROOT, "christine-resnet18", "results")
os.makedirs(RESULTS, exist_ok=True)

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)
EPOCHS, BATCH_SIZE, LR = 12, 64, 1e-4

torch.manual_seed(SEED); np.random.seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)


device: cuda


In [ ]:
# CIFAR-10 download (one-time, ~170MB) — skipped if already present.
import tarfile, urllib.request
_DATA = os.path.join(PROJECT_ROOT, "data")
_READY = os.path.join(_DATA, "cifar-10-batches-py", "test_batch")
os.makedirs(_DATA, exist_ok=True)
if not os.path.isfile(_READY):
    print("Downloading CIFAR-10 (~170MB)...")
    _tgz = os.path.join(_DATA, "cifar-10-python.tar.gz")
    urllib.request.urlretrieve(
        "https://www.cs.toronto.edu/~kriz/cifar-10-python.tar.gz", _tgz)
    print("Extracting...")
    with tarfile.open(_tgz, "r:gz") as _tf:
        for _m in _tf.getmembers():
            if _m.isfile():  # manual copy: avoids tar ownership quirks
                _out = os.path.join(_DATA, _m.name)
                os.makedirs(os.path.dirname(_out), exist_ok=True)
                with open(_out, 'wb') as _f:
                    _f.write(_tf.extractfile(_m).read())
    print("CIFAR-10 ready.")
else:
    print("CIFAR-10 already present.")


Extracting...
CIFAR-10 ready.


## 1. Data

Canonical **45k train / 5k val / 10k test** splits — stratified, seed 0,
identical for all three methods (`shared/data_utils.py`).
ResNet-18 expects 224×224 inputs with **ImageNet normalization**
(the "correct preprocessing" rubric point). Train augmentation:
horizontal flip.


In [ ]:
class CifarDataset(Dataset):
    """Wraps uint8 numpy images so transforms + DataLoader can use them."""
    def __init__(self, images, labels, transform):
        self.images = images
        self.labels = labels
        self.transform = transform
    def __len__(self):
        return len(self.images)
    def __getitem__(self, i):
        return self.transform(self.images[i]), int(self.labels[i])

print('Loading canonical CIFAR-10 splits...')
Xtr, ytr, Xva, yva, Xte, yte = load_canonical_splits()
print(f'train {Xtr.shape}, val {Xva.shape}, test {Xte.shape}')

train_tf = transforms.Compose([
    transforms.ToPILImage(),
        transforms.Resize(224),
        transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
eval_tf = transforms.Compose([
    transforms.ToPILImage(),
        transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
train_loader = DataLoader(CifarDataset(Xtr, ytr, train_tf),
                          batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=(device == 'cuda'))
val_loader = DataLoader(CifarDataset(Xva, yva, eval_tf),
                        batch_size=256, num_workers=2)
test_loader = DataLoader(CifarDataset(Xte, yte, eval_tf),
                         batch_size=256, num_workers=2)
print('loaders ready')


Loading canonical CIFAR-10 splits...
train (45000, 32, 32, 3), val (5000, 32, 32, 3), test (10000, 32, 32, 3)
loaders ready


## 2. Model

ImageNet-pretrained ResNet-18 from `resnet18_finetune.py` (~11.7M params)
with the final layer replaced for 10 CIFAR-10 classes.
First run downloads the ImageNet weights (~45MB) automatically.


In [ ]:
model = build_model(num_classes=10, pretrained=True).to(device)
n_params = count_parameters(model)
print(f"{n_params:,} trainable parameters ({n_params/1e6:.2f}M)")


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 184MB/s]


11,181,642 trainable parameters (11.18M)


## 3. Training

Adam at a **small lr (1e-4)** — the pretrained filters are already good,
so we nudge them instead of relearning. 12 epochs; pretrained nets
converge fast.
The best-*validation* checkpoint is kept automatically — never pick by test.


In [ ]:
print(f"Training ({EPOCHS} epochs)...")
log = []
t0 = time.perf_counter()
history = fit(model, train_loader, val_loader, device,
              epochs=EPOCHS, lr=LR, verbose=True)
train_time_min = (time.perf_counter() - t0) / 60
log.append(f"training_time_minutes: {train_time_min:.1f}")
print(f"Training took {train_time_min:.1f} min")
torch.save(model.state_dict(), os.path.join(RESULTS, "model.pt"))
plot_curves(history, os.path.join(RESULTS, "training_curves.png"))
print("saved model.pt + training_curves.png")


Training (12 epochs)...
  epoch  1/12  train loss 0.3315 acc 0.8909  |  val loss 0.1998 acc 0.9300
  epoch  2/12  train loss 0.1355 acc 0.9554  |  val loss 0.2018 acc 0.9300
  epoch  3/12  train loss 0.0826 acc 0.9724  |  val loss 0.1787 acc 0.9438
  epoch  4/12  train loss 0.0535 acc 0.9830  |  val loss 0.1642 acc 0.9474
  epoch  5/12  train loss 0.0362 acc 0.9892  |  val loss 0.1666 acc 0.9454
  epoch  6/12  train loss 0.0243 acc 0.9934  |  val loss 0.1544 acc 0.9522
  epoch  7/12  train loss 0.0145 acc 0.9966  |  val loss 0.1397 acc 0.9580
  epoch  8/12  train loss 0.0075 acc 0.9983  |  val loss 0.1383 acc 0.9580
  epoch  9/12  train loss 0.0061 acc 0.9986  |  val loss 0.1410 acc 0.9584
  epoch 10/12  train loss 0.0032 acc 0.9996  |  val loss 0.1392 acc 0.9612
  epoch 11/12  train loss 0.0026 acc 0.9997  |  val loss 0.1333 acc 0.9616
  epoch 12/12  train loss 0.0018 acc 0.9998  |  val loss 0.1312 acc 0.9628
  best val accuracy: 0.9628
Training took 30.4 min
saved model.pt + training

## 4. Evaluation (run once on the test set)

Top-1 accuracy, macro F1, confusion matrix, ms/image, parameter count,
training time → `metrics.json` (same schema as the other two methods).


In [ ]:
criterion = torch.nn.CrossEntropyLoss()
model.eval()
# Honest ms/image: time the forward pass only
t0 = time.perf_counter()
with torch.no_grad():
    for images, _ in test_loader:
        _ = model(images.to(device))
        if device == "cuda":
            torch.cuda.synchronize()
ms_per_image = (time.perf_counter() - t0) / len(Xte) * 1000
print(f"{ms_per_image:.2f} ms/image ({device})")

_, _, y_pred, y_true = evaluate(model, test_loader, criterion, device)
cm = confusion_matrix(y_true, y_pred)
acc, macro_f1, per_class_f1 = accuracy_and_macro_f1(cm)
print(f"Top-1 accuracy: {acc*100:.2f}%")
print(f"Macro F1: {macro_f1:.4f}")
log.append(f"top1_accuracy_pct: {acc*100:.2f}")
log.append(f"macro_f1: {macro_f1:.4f}")

metrics = {
    "method": "ResNet-18 (ImageNet pretrained, fine-tuned)",
    "dataset": "CIFAR-10",
    "n_train": int(len(Xtr)),
    "n_val": int(len(Xva)),
    "n_test": int(len(Xte)),
    "top1_accuracy_pct": round(acc*100, 2),
    "macro_f1": round(macro_f1, 4),
    "per_class_f1": {n: round(float(v), 4) for n, v in zip(CLASS_NAMES, per_class_f1)},
    "runtime_ms_per_image": round(ms_per_image, 2),
    "runtime_device": device,
    "parameter_count_millions": round(n_params/1e6, 2),
    "training_time_minutes": round(train_time_min, 1),
    "hyperparams": {"epochs": EPOCHS, "batch_size": BATCH_SIZE, "lr": LR, "optimizer": "Adam", "weight_decay": 1e-4, "scheduler": "cosine", "preprocessing": "resize224+imagenet_norm", "augmentation": "flip"},
}
with open(os.path.join(RESULTS, "metrics.json"), "w") as f:
    json.dump(metrics, f, indent=2)
np.save(os.path.join(RESULTS, "confusion_matrix.npy"), cm)
plot_confusion_matrix(cm, CLASS_NAMES,
                      f"ResNet-18 confusion matrix (acc {acc*100:.2f}%)",
                      os.path.join(RESULTS, "confusion_matrix.png"))
with open(os.path.join(RESULTS, "training_log.txt"), "w") as f:
    f.write("\n".join(log) + "\n")
print("Done — metrics.json, confusion_matrix.png, training_log.txt saved to", RESULTS)


2.22 ms/image (cuda)
Top-1 accuracy: 96.30%
Macro F1: 0.9630
Done — metrics.json, confusion_matrix.png, training_log.txt saved to /content/prism/christine-resnet18/results


In [ ]:
import json

path = "/content/prism/christine-resnet18/results/metrics.json"
with open(path) as f:
    metrics = json.load(f)

for name, score in metrics["per_class_f1"].items():
    print(f"{name:10s} {score:.4f}")

airplane   0.9725
automobile 0.9786
bird       0.9565
cat        0.9219
deer       0.9625
dog        0.9333
frog       0.9786
horse      0.9764
ship       0.9765
truck      0.9729
